[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cursuri-inf/DCTC/blob/main/notebooks/explorare-sablon.ipynb)

# Explorarea numerică a temei: șablonul DCTC

Șablonul temelor 7–10 ale proiectului, cu date generate în notebook: șirul Collatz, numerele prime, metoda Monte Carlo, metodele lui Newton și a bisecției. Notebook-ul generează datele temei alese, calculează câteva statistici, desenează două grafice și le salvează ca PNG pentru raport, împreună cu tabelul datelor generate (CSV).

Cum îl folosești:

1. *File > Save a copy in Drive*: lucrezi pe copia ta, șablonul rămâne neschimbat.
2. Alegi tema în celula de parametri (secțiunea 1), apoi *Runtime > Run all*; cu valorile implicite rulează în mai puțin de un minut.
3. Schimbi parametrii și textele marcate „De completat”; generatoarele (secțiunea 3) rămân cum sunt.
4. Înainte de a salva în GitHub: *Runtime > Restart session and run all*, ca rezultatele salvate să fie ale unei rulări de la zero.

În notebook nu intră parole, chei sau căi de pe calculatorul tău.

## 1. Parametrii

`TEMA` alege generatorul. Parametrii care contează, pe teme: Collatz, `N` (cel mai mare $n$) și `N_MIN` (de unde începe intervalul studiat); numerele prime, `N` (limita căutării) și `N_MIN`; Monte Carlo, `N` (numărul de încercări) și `SAMANTA`, care face ca rezultatele aleatoare să fie aceleași la fiecare rulare; Newton și bisecția, `FUNCTIA` și `TOLERANTA`.

In [ ]:
# Parametrii explorării
TEMA = "collatz"            # "collatz", "prime", "monte_carlo" sau "newton_bisectie"
N = 100_000                 # Collatz: n până la N; prime: x până la N; Monte Carlo: numărul de încercări
N_MIN = 1                   # Collatz și prime: începutul intervalului din statistici și grafice
SAMANTA = 12345             # Monte Carlo: sămânța generatorului de numere aleatoare
FUNCTIA = 3                 # Newton și bisecție: funcția din graficul de evoluție (1, 2 sau 3)
TOLERANTA = 1e-10           # Newton și bisecție: precizia cerută
FOLDER_REZULTATE = "rezultate"  # unde se salvează PNG-urile și CSV-ul
DESCARCA_FIGURI = False     # True: la final, Colab descarcă fișierele pe calculator

## 2. Bibliotecile

numpy face calculele pe tablouri de numere, pandas ține tabelul rezultatelor, matplotlib desenează graficele; în Colab sunt deja instalate.

In [ ]:
import math
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CULOARE_PRINCIPAL = "#9B1B30"    # seria despre care vorbim
CULOARE_REFERINTA = "#161616"    # valoarea de referință
CULOARE_REST = "#8A8A8A"         # celelalte serii
print("numpy", np.__version__, "· pandas", pd.__version__, "· matplotlib", matplotlib.__version__)

## 3. Generatoarele

Câte o funcție pentru fiecare temă; fiecare întoarce tabelul datelor generate. Nu este nevoie să le modifici.

In [ ]:
def collatz(n_max):
    """Lungimea traiectoriei (numărul de pași până la 1) pentru fiecare n de la 1 la n_max."""
    lungime = np.zeros(n_max + 1, dtype=np.int64)
    for n in range(2, n_max + 1):
        x, pasi = n, 0
        while x >= n:  # până coboară sub n, unde lungimea este deja cunoscută
            x = x // 2 if x % 2 == 0 else 3 * x + 1
            pasi += 1
        lungime[n] = pasi + lungime[x]
    return pd.DataFrame({"n": np.arange(1, n_max + 1), "lungime": lungime[1:]})


def prime(x_max, x_min=10):
    """π(x) din ciurul lui Eratostene, comparat cu x / ln x și cu Li(x), în 200 de puncte între x_min și x_max."""
    ciur = np.ones(x_max + 1, dtype=bool)
    ciur[:2] = False
    for p in range(2, int(x_max ** 0.5) + 1):
        if ciur[p]:
            ciur[p * p::p] = False
    pi = np.cumsum(ciur)
    f = 1 / np.log(np.arange(2, x_max + 1, dtype=float))
    li = np.concatenate([[0.0, 0.0], np.cumsum(f) - (f[0] + f) / 2])  # Li(x), regula trapezelor cu pasul 1
    x_min = max(10, x_min)
    puteri = [10 ** k for k in range(1, 10) if x_min <= 10 ** k <= x_max]
    x = np.unique(np.concatenate([np.geomspace(x_min, x_max, 200).astype(int), puteri]))
    return pd.DataFrame({"x": x, "pi": pi[x], "x_ln_x": x / np.log(x), "Li": li[x]})


def monte_carlo(incercari, samanta):
    """Estimarea lui π cu puncte aleatoare în pătratul unitate, după fiecare încercare."""
    rng = np.random.default_rng(samanta)
    x, y = rng.random(incercari), rng.random(incercari)
    in_cerc = np.cumsum(x * x + y * y <= 1)
    k = np.arange(1, incercari + 1)
    estimare = 4 * in_cerc / k
    return pd.DataFrame({"incercari": k, "estimare": estimare, "eroare": np.abs(estimare - math.pi)})


FUNCTII = {  # f, derivata, intervalul pentru bisecție, punctul de start pentru Newton
    1: ("x² − 2", lambda x: x * x - 2, lambda x: 2 * x, (1.0, 2.0), 2.0),
    2: ("cos x − x", lambda x: math.cos(x) - x, lambda x: -math.sin(x) - 1, (0.0, 1.0), 1.0),
    3: ("x³ − 2x − 5", lambda x: x ** 3 - 2 * x - 5, lambda x: 3 * x * x - 2, (2.0, 3.0), 2.0),
}


def newton_bisectie(toleranta, max_iteratii=100):
    """Eroarea la fiecare iterație, pentru Newton și bisecție, pe cele trei funcții."""
    randuri = []
    for nr, (nume, f, df, (a, b), x0) in FUNCTII.items():
        radacina = x0
        for _ in range(100):  # referința: Newton dus până la precizia mașinii
            radacina -= f(radacina) / df(radacina)
        x = x0
        for k in range(1, max_iteratii + 1):
            x -= f(x) / df(x)
            randuri.append((nr, nume, "Newton", k, abs(x - radacina)))
            if abs(x - radacina) < toleranta:
                break
        for k in range(1, max_iteratii + 1):
            m = (a + b) / 2
            a, b = (a, m) if f(a) * f(m) <= 0 else (m, b)
            randuri.append((nr, nume, "bisecție", k, abs((a + b) / 2 - radacina)))
            if abs((a + b) / 2 - radacina) < toleranta:
                break
    return pd.DataFrame(randuri, columns=["functia", "nume", "metoda", "iteratia", "eroare"])

## 4. Datele generate

Se rulează generatorul temei alese; timpul de calcul este afișat.

In [ ]:
start = time.perf_counter()
if TEMA == "collatz":
    date = collatz(N)
    date = date[date["n"] >= N_MIN].reset_index(drop=True)
elif TEMA == "prime":
    date = prime(N, N_MIN)
elif TEMA == "monte_carlo":
    date = monte_carlo(N, SAMANTA)
elif TEMA == "newton_bisectie":
    date = newton_bisectie(TOLERANTA)
else:
    raise ValueError(f"Tema necunoscută: {TEMA}")
print(f"Tema {TEMA}: {len(date)} rânduri, în {time.perf_counter() - start:.1f} s")
date.head()

## 5. Statistici

Câteva valori care rezumă datele temei; ele intră în tabelul din raport.

In [ ]:
if TEMA == "collatz":
    rezumat = date["lungime"].describe().round(2)
    print("Cea mai lungă traiectorie:", int(date.loc[date["lungime"].idxmax(), "n"]),
          "cu", int(date["lungime"].max()), "pași")
elif TEMA == "prime":
    ultim = date.iloc[-1]
    rezumat = pd.Series({"x": ultim["x"], "π(x)": ultim["pi"],
                         "eroarea relativă x / ln x (%)": (ultim["pi"] - ultim["x_ln_x"]) / ultim["pi"] * 100,
                         "eroarea relativă Li(x) (%)": (ultim["Li"] - ultim["pi"]) / ultim["pi"] * 100}).round(4)
elif TEMA == "monte_carlo":
    puteri = [10 ** k for k in range(1, int(math.log10(N)) + 1)]
    rezumat = date.set_index("incercari").loc[puteri, ["estimare", "eroare"]].round(5)
else:
    rezumat = date.groupby(["nume", "metoda"])["iteratia"].max().unstack()
    rezumat.columns.name = f"iterații până la {TOLERANTA:g}"
rezumat

## 6. Graficul de evoluție

Cum se schimbă mărimea studiată pe măsură ce cresc $n$, $x$, numărul de încercări sau al iterațiilor: lungimea traiectoriei (Collatz), eroarea relativă a celor două aproximări (prime), estimarea lui π (Monte Carlo), eroarea la fiecare iterație (Newton și bisecție). Seria despre care vorbim este colorată.

In [ ]:
Path(FOLDER_REZULTATE).mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
if TEMA == "collatz":
    ax.scatter(date["n"], date["lungime"], s=1, color=CULOARE_REST, alpha=0.4)
    ax.plot(date["n"], date["lungime"].cummax(), color=CULOARE_PRINCIPAL, linewidth=2)
    ax.annotate("maximul de până la n", (date["n"].iloc[-1], date["lungime"].max()),
                xytext=(-5, 8), textcoords="offset points", ha="right", color=CULOARE_PRINCIPAL)
    ax.set_xlabel("n")
    ax.set_ylabel("lungimea traiectoriei (pași)")
elif TEMA == "prime":
    for col, eticheta, culoare in (("x_ln_x", "x / ln x", CULOARE_PRINCIPAL), ("Li", "Li(x)", CULOARE_REST)):
        eroare = (date[col] - date["pi"]).abs() / date["pi"] * 100
        ax.plot(date["x"], eroare, color=culoare, linewidth=2 if col == "x_ln_x" else 1.4)
        ax.annotate(eticheta, (date["x"].iloc[-1], eroare.iloc[-1]), xytext=(5, 0),
                    textcoords="offset points", va="center", color=culoare)
    ax.set_xscale("log")
    ax.set_xlabel("x")
    ax.set_ylabel("eroarea relativă față de π(x) (%)")
elif TEMA == "monte_carlo":
    ax.plot(date["incercari"], date["estimare"], color=CULOARE_PRINCIPAL, linewidth=1.5)
    ax.axhline(math.pi, color=CULOARE_REFERINTA, linewidth=1, linestyle="--")
    ax.annotate("π", (date["incercari"].iloc[-1], math.pi), xytext=(5, 0), textcoords="offset points",
                va="center", color=CULOARE_REFERINTA)
    ax.set_xscale("log")
    ax.set_xlabel("numărul de încercări")
    ax.set_ylabel("estimarea lui π")
else:
    alese = date[date["functia"] == FUNCTIA]
    for metoda, culoare in (("Newton", CULOARE_PRINCIPAL), ("bisecție", CULOARE_REST)):
        serie = alese[alese["metoda"] == metoda]
        ax.plot(serie["iteratia"], serie["eroare"].clip(lower=1e-17), marker="o", markersize=3, color=culoare)
        ax.annotate(metoda, (serie["iteratia"].iloc[-1], max(serie["eroare"].iloc[-1], 1e-17)),
                    xytext=(5, 0), textcoords="offset points", va="center", color=culoare)
    ax.set_yscale("log")
    ax.set_xlabel("iterația")
    ax.set_ylabel(f"eroarea, pentru f(x) = {FUNCTII[FUNCTIA][0]}")
ax.set_title("Evoluția", loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{FOLDER_REZULTATE}/evolutie.png", dpi=200)
plt.show()

## 7. Graficul de comparație

O comparație pe categorii: distribuția lungimilor (Collatz), π(x) și cele două aproximări la 10, 100, 1000… (prime), eroarea la 10, 100, 1000… încercări (Monte Carlo), iterațiile necesare fiecărei metode (Newton și bisecție).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
if TEMA == "collatz":
    ax.hist(date["lungime"], bins=60, color=CULOARE_REST, edgecolor="white")
    ax.axvline(date["lungime"].mean(), color=CULOARE_PRINCIPAL, linewidth=2)
    ax.set_xlabel("lungimea traiectoriei (pași)")
    ax.set_ylabel("câte numere n")
elif TEMA == "prime":
    puncte = date[date["x"].isin([10 ** k for k in range(1, 10)])]
    eticheta = [f"{int(v):,}".replace(",", ".") for v in puncte["x"]]
    pozitii = np.arange(len(puncte))
    ax.bar(pozitii - 0.27, puncte["pi"], 0.27, color=CULOARE_REFERINTA)
    ax.bar(pozitii, puncte["x_ln_x"], 0.27, color=CULOARE_PRINCIPAL)
    ax.bar(pozitii + 0.27, puncte["Li"], 0.27, color=CULOARE_REST)
    ax.set_xticks(pozitii, eticheta)
    ax.set_yscale("log")
    ax.set_xlabel("x")
    ax.set_ylabel("π(x) negru, x / ln x colorat, Li(x) gri")
elif TEMA == "monte_carlo":
    puteri = [10 ** k for k in range(1, int(math.log10(N)) + 1)]
    erori = date.set_index("incercari").loc[puteri, "eroare"]
    ax.bar([f"{p:,}".replace(",", ".") for p in puteri], erori.values, color=CULOARE_REST)
    ax.plot(range(len(puteri)), [1 / math.sqrt(p) for p in puteri], color=CULOARE_PRINCIPAL, marker="o")
    ax.set_yscale("log")
    ax.set_xlabel("numărul de încercări")
    ax.set_ylabel("eroarea (bare); 1/√n (linia)")
else:
    iteratii = date.groupby(["nume", "metoda"])["iteratia"].max().unstack()
    pozitii = np.arange(len(iteratii))
    ax.barh(pozitii - 0.2, iteratii["Newton"], 0.4, color=CULOARE_PRINCIPAL)
    ax.barh(pozitii + 0.2, iteratii["bisecție"], 0.4, color=CULOARE_REST)
    ax.set_yticks(pozitii, iteratii.index)
    ax.set_xlabel(f"iterații până la eroarea {TOLERANTA:g}: Newton colorat, bisecția gri")
ax.set_title("Comparația", loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{FOLDER_REZULTATE}/comparatie.png", dpi=200)
plt.show()

## 8. Formula temei

Păstrează formula temei tale și șterge-le pe celelalte.

- Collatz: $f(n) = n/2$ dacă $n$ este par și $f(n) = 3n + 1$ dacă $n$ este impar; lungimea traiectoriei este numărul de aplicări ale lui $f$ până la 1.
- Numerele prime: eroarea relativă este $\left|a(x) - \pi(x)\right| / \pi(x)$, pentru aproximarea $a(x)$; teorema numerelor prime, $\pi(x) \sim \dfrac{x}{\ln x}$, și integrala logaritmică $\mathrm{Li}(x) = \displaystyle\int_2^x \frac{dt}{\ln t}$.
- Monte Carlo: $\pi \approx 4\,\dfrac{k}{n}$, unde $k$ este numărul punctelor din sfertul de cerc; eroarea scade ca $1/\sqrt{n}$.
- Newton și bisecția: $x_{k+1} = x_k - \dfrac{f(x_k)}{f'(x_k)}$; bisecția înjumătățește intervalul la fiecare pas, deci eroarea după $k$ pași este cel mult $(b - a)/2^{k}$.

## 9. Concluzia

De completat: 3–5 propoziții care răspund la întrebarea studiului, cu valori concrete din tabel și din grafice, și o limită a explorării.

## 10. Fișierele pentru raport

PNG-urile și tabelul datelor generate (`date-generate.csv`) sunt în folderul `rezultate` al sesiunii Colab (panoul *Files*, în stânga) și dispar când sesiunea se închide. Le descarci (clic dreapta pe fișier, *Download*, sau `DESCARCA_FIGURI = True`) și le pui în depozitul echipei: figurile în `report/figures/`, tabelul în `data/`, printr-un pull request.

In [ ]:
date.to_csv(f"{FOLDER_REZULTATE}/date-generate.csv", index=False)
fisiere = sorted(Path(FOLDER_REZULTATE).glob("*.*"))
print("Fișierele salvate:", [f.name for f in fisiere])
if DESCARCA_FIGURI:
    try:
        from google.colab import files
        for f in fisiere:
            files.download(str(f))
    except ImportError:
        print("Notebook-ul nu rulează în Colab; fișierele sunt în folderul", FOLDER_REZULTATE)